[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onuralpszr/litert-llm-cookbook/blob/main/colab/15_session_api.ipynb)

# Example 15: Session API

Use the low-level Session API (create_session, run_prefill, run_decode, run_decode_async) instead of Conversation. Also demonstrates run_text_scoring to rank candidates by log-likelihood.

In [ ]:
!pip install litert-lm-api-nightly litert-lm -q

In [ ]:
import os
import subprocess

model_url = 'https://huggingface.co/litert-community/gemma-4-E4B-it-litert-lm/resolve/main/gemma-4-E4B-it.litertlm?download=true'
output_path = '/content/gemma-4-E4B-it.litertlm'

if not os.path.exists(output_path):
    subprocess.run(['curl', '-L', model_url, '-o', output_path], check=True)
    print(f'Model downloaded to {output_path}')
else:
    print(f'Model already exists at {output_path}')

In [ ]:
import litert_lm

MODEL_PATH = "/content/gemma-4-E4B-it.litertlm"

litert_lm.set_min_log_severity(litert_lm.LogSeverity.ERROR)

_BOS = "<start_of_turn>"
_EOS = "<end_of_turn>"


def _prompt(text: str) -> str:
    return f"{_BOS}user\n{text}{_EOS}\n{_BOS}model\n"


with litert_lm.Engine(MODEL_PATH) as engine:
    print("=== Part A: generation ===")
    with engine.create_session() as session:
        session.run_prefill([_prompt("What is the capital of Japan? Answer in one word.")])
        result = session.run_decode()
        print(f"Response: {result.texts[0].replace(_EOS, '').strip()}\n")

    print("=== Part B: streaming ===")
    with engine.create_session() as session:
        session.run_prefill([_prompt("Count from 1 to 5, one number per line.")])
        print("Response: ", end="", flush=True)
        buf = ""
        printed = 0
        for chunk in session.run_decode_async():
            buf += chunk.texts[0]
            if _EOS in buf:
                print(buf[printed : buf.index(_EOS)], end="", flush=True)
                break
            safe = len(buf) - len(_EOS) + 1
            if safe > printed:
                print(buf[printed:safe], end="", flush=True)
                printed = safe
        print("\n")

    print("=== Part C: text scoring ===")
    context = "Paris is the capital of"
    candidates = [" France.", " Germany.", " Italy.", " Spain."]

    scored = []
    for candidate in candidates:
        with engine.create_session() as session:
            session.run_prefill([context])
            result = session.run_text_scoring([candidate])
            scored.append((candidate, result.scores[0]))

    ranked = sorted(scored, key=lambda x: x[1], reverse=True)
    print(f"Context: {context!r}")
    print("Candidate scores (higher = more likely):")
    for text, score in ranked:
        print(f"  {score:+.4f}  {text.strip()!r}")